# COMP9517 — iNat2021 Colab Quickstart (契约版)
契约驱动仓库,所有命令从**仓库根**以 `python -m src...` 跑。
数据放 Drive,每次 session 把 `images_256/` 拷到 Colab 本地盘再训。

## 0. 环境 + 拉仓库

In [ ]:
!nvidia-smi -L
# 换成你们组的私库地址
!git clone https://github.com/<your-group>/COMP9517-GroupProject.git
%cd COMP9517-GroupProject
!pip -q install -r requirements.txt

## 1. 数据(A 做一次,传 Drive;其他人只挂载)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
# A 首次构建(需已解压 train_mini/ val/ + json):
# !python -m src.data.build_subset --inat_root /content/inat2021 --out data \
#     --n_classes 500 --n_train 40 --n_val 10 --n_test 10 --seed 42
# 其他人:把 Drive 上的 data/ 软链或拷进来
!ln -sfn /content/drive/MyDrive/COMP9517_data data
# M1 泄漏+计数校验(随机预测 top1 应 ≈0.2%)
!python -c "from src.common.manifest import validate_split; print(validate_split('data')[1])" 

## 2. [D] 迁移学习(预训练,出最佳模型)

In [ ]:
!python -m src.methods.transfer.run --arch resnet50 --tag res50_ft --epochs 30

## 3. [C] 从零训练 + [B] 传统管线

In [ ]:
!python -m src.methods.scratch.run --arch resnet50 --tag res50_scr --epochs 60
!python -m src.methods.traditional.bovw --k 512 --tag sift512

## 4. [A] 汇总表 + 误差分析

In [ ]:
!python -m src.eval.aggregate --results results
import glob; run=sorted(glob.glob('predictions/*__test.npz'))[0]
!python -m src.eval.analysis --npz "$run" --classes data/classes_500.json --out figures/confusion.png

## 5. [E] Advanced ②:退化鲁棒性扫描(你的主场)

In [ ]:
!python -m src.advanced.robustness.run --method deep --arch resnet50 \
    --ckpt data/checkpoints/transfer_resnet50__res50_ft/best.pt --tag res50_ft
# 画曲线:读 results/*__<corruption>_s*__test.json 的 top1/macro_f1 vs severity
import glob,json,matplotlib.pyplot as plt,collections,re
curves=collections.defaultdict(list)
for p in sorted(glob.glob('results/*_s*__test.json')):
    r=json.load(open(p)); d=r['degradation']; curves[d['type']].append((d['severity'],r['metrics']['top1']))
for name,pts in curves.items():
    pts=sorted(pts); plt.plot([s for s,_ in pts],[t for _,t in pts],marker='o',label=name)
plt.xlabel('severity'); plt.ylabel('top-1'); plt.legend(); plt.title('Robustness'); plt.show()

## 6. [E] Advanced ①:Grad-CAM

In [ ]:
!python -m src.advanced.gradcam.gradcam --arch resnet50 \
    --ckpt data/checkpoints/transfer_resnet50__res50_ft/best.pt --out figures/gradcam.png
from IPython.display import Image; Image('figures/gradcam.png')